In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

# Fungsi bantu: muat ARFF dan ubah target Y/N menjadi 1/0
def load_nasa_arff(filepath):
    data, _ = arff.loadarff(filepath)
    df = pd.DataFrame(data)
    target_col = df.columns[-1]
    y = df[target_col].apply(lambda x: 1 if (x.decode("utf-8") if isinstance(x, bytes) else x) == "Y" else 0)
    X = df.drop(columns=[target_col]).apply(pd.to_numeric)
    return X, y

# Fungsi bantu: hitung Recall@20% LOC
def recall_at_20_loc(y_true, y_prob, loc_values, budget_ratio=0.20):
    eval_df = pd.DataFrame({"y": y_true.values, "prob": y_prob, "loc": loc_values.values})
    eval_df = eval_df.sort_values(by="prob", ascending=False).reset_index(drop=True)
    eval_df["cum_loc"] = eval_df["loc"].cumsum()
    
    budget_loc = eval_df["loc"].sum() * budget_ratio
    inspected = eval_df[eval_df["cum_loc"] <= budget_loc]
    
    total_defects = eval_df["y"].sum()
    return inspected["y"].sum() / total_defects if total_defects > 0 else 0.0

data_dir = Path("../data")
datasets = ["JM1.arff", "PC4.arff", "MC1.arff"]
results = []

for fname in datasets:
    X, y = load_nasa_arff(data_dir / fname)
    
    # Split 60% Train, 20% Val, 20% Test (Stratified, Seed=42)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
    )
    
    models = {
        "Baseline_LR": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "Baseline_RF": RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    for name, model in models.items():
        model.fit(X_train, y_train)
        
        val_prob = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]
        test_pred = model.predict(X_test)
        
        results.append({
            "Dataset": fname.replace(".arff", ""),
            "Model": name,
            "Val_PR_AUC": round(average_precision_score(y_val, val_prob), 4),
            "Test_PR_AUC": round(average_precision_score(y_test, test_prob), 4),
            "Recall_at_20_LOC": round(recall_at_20_loc(y_test, test_prob, X_test["LOC_TOTAL"]), 4),
            "Test_Precision": round(precision_score(y_test, test_pred, zero_division=0), 4),
            "Test_Recall": round(recall_score(y_test, test_pred, zero_division=0), 4),
            "Test_F1": round(f1_score(y_test, test_pred, zero_division=0), 4)
        })

res_df = pd.DataFrame(results)
res_df.to_csv("../results/baseline_metrics.csv", index=False)
print(res_df.to_string(index=False))

Dataset       Model  Val_PR_AUC  Test_PR_AUC  Recall_at_20_LOC  Test_Precision  Test_Recall  Test_F1
    JM1 Baseline_LR      0.3783       0.3830            0.1164          0.5741       0.0925   0.1594
    JM1 Baseline_RF      0.4243       0.3657            0.1552          0.4919       0.1821   0.2658
    PC4 Baseline_LR      0.6151       0.6663            0.3889          0.6154       0.4444   0.5161
    PC4 Baseline_RF      0.6569       0.6899            0.4167          0.8125       0.3611   0.5000
    MC1 Baseline_LR      0.1250       0.0450            0.1111          0.0000       0.0000   0.0000
    MC1 Baseline_RF      0.3116       0.0829            0.3333          0.0000       0.0000   0.0000
